# 04 CNN 图像分类

## 本节目标

本节学习 CNN 如何处理图像数据。

你将完成：

- 理解图像输入 `[N, C, H, W]`。
- 使用卷积、激活、池化和全连接层搭建小型 CNN。
- 在 digits 小型图像数据集上训练分类模型。
- 绘制 loss 曲线、查看预测样例和中间特征图。
- 总结 CNN 中最常见的 shape 问题。

## 背景问题

全连接网络可以处理图像，但直接 flatten 会削弱图像的空间结构，也会带来较多参数。CNN 使用卷积核在局部区域滑动，通过参数共享学习图像模式。

本节关注的问题是：卷积层如何把图像变成更适合分类的特征表示？

## 核心概念

- **Channel**：图像通道，灰度图为 1，RGB 图为 3。
- **Conv2d**：二维卷积层，用局部窗口学习空间模式。
- **MaxPool2d**：池化层，降低空间尺寸。
- **Feature map**：卷积层输出的特征图。
- **Flatten**：把特征图展开为分类头需要的一维向量。

调试 CNN 时，最重要的是每一层之后的 shape。

## 最小代码示例：加载图像数据

这里使用 scikit-learn 内置 digits 数据集，图像大小为 `8x8`，适合快速实验，不需要下载外部数据。

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

torch.manual_seed(42)
np.random.seed(42)

digits = load_digits()
images = digits.images.astype(np.float32) / 16.0
labels = digits.target.astype(np.int64)

x_train, x_test, y_train, y_test = train_test_split(
    images, labels, test_size=0.2, random_state=42, stratify=labels
)

x_train = torch.tensor(x_train).unsqueeze(1)
x_test = torch.tensor(x_test).unsqueeze(1)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

print("x_train shape:", x_train.shape)
print("y_train shape:", y_train.shape)

## 最小代码示例：可视化样本

先看数据，再建模型。这样可以确认任务确实是手写数字分类，而不是在抽象 shape 上盲写代码。

In [ ]:
plt.figure(figsize=(7, 2))
for i in range(8):
    plt.subplot(1, 8, i + 1)
    plt.imshow(x_train[i, 0], cmap="gray")
    plt.title(str(y_train[i].item()))
    plt.axis("off")
plt.tight_layout()
plt.show()

## 最小代码示例：卷积后的 shape

下面先单独跑一个卷积和池化，观察空间尺寸如何变化。

In [ ]:
sample = x_train[:4]
conv = nn.Conv2d(1, 8, kernel_size=3, padding=1)
pool = nn.MaxPool2d(2)

with torch.no_grad():
    conv_out = conv(sample)
    pool_out = pool(torch.relu(conv_out))

print("input:", sample.shape)
print("after conv:", conv_out.shape)
print("after pool:", pool_out.shape)

## 完整实验：定义 CNN

digits 图像较小，所以模型保持轻量。两次池化后，`8x8` 会变成 `2x2`，分类头输入维度是 `16 * 2 * 2`。

In [ ]:
class TinyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(8, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * 2 * 2, 32),
            nn.ReLU(),
            nn.Linear(32, 10),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


model = TinyCNN()
print(model)

## 完整实验：训练循环

这里使用 mini-batch 训练。与前面章节相比，训练循环没有本质变化，只是输入从二维点变成了四维图像张量。

In [ ]:
train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=64, shuffle=True)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

losses = []

for epoch in range(25):
    model.train()
    total_loss = 0.0

    for batch_x, batch_y in train_loader:
        logits = model(batch_x)
        loss = criterion(logits, batch_y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch_x.size(0)

    losses.append(total_loss / len(train_loader.dataset))

model.eval()
with torch.no_grad():
    test_logits = model(x_test)
    test_pred = test_logits.argmax(dim=1)
    test_acc = (test_pred == y_test).float().mean().item()

print(f"final loss={losses[-1]:.4f}")
print(f"test accuracy={test_acc:.3f}")

## 实验观察：loss 曲线和预测样例

从运行结果可以看到，loss 通常会明显下降。digits 数据比较小，轻量 CNN 也能学到有效模式。

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(losses)
plt.title("CNN training loss")
plt.xlabel("Epoch")
plt.ylabel("Cross entropy")
plt.grid(alpha=0.3)
plt.show()

plt.figure(figsize=(9, 3))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_test[i, 0], cmap="gray")
    plt.title(f"p:{test_pred[i].item()} y:{y_test[i].item()}")
    plt.axis("off")
plt.tight_layout()
plt.show()

## 实验观察：中间特征图

特征图不是最终答案，但可以帮助我们观察卷积层是否产生了不同响应。

In [ ]:
model.eval()
with torch.no_grad():
    feature_maps = model.features[:2](x_test[:1])

plt.figure(figsize=(8, 3))
for i in range(8):
    plt.subplot(2, 4, i + 1)
    plt.imshow(feature_maps[0, i].numpy(), cmap="viridis")
    plt.axis("off")
    plt.title(f"map {i}")
plt.tight_layout()
plt.show()

## 常见错误

1. **缺少 channel 维度**：卷积层期望 `[N, C, H, W]`。  
2. **Linear 输入维度算错**：池化后尺寸变化没有跟上。  
3. **标签 dtype 错误**：分类标签应为 `torch.long`。  
4. **评估时忘记 `model.eval()`**：影响 Dropout/BatchNorm 行为。  
5. **评估时忘记 `torch.no_grad()`**：浪费内存和计算。

调试时可以在 `forward()` 中临时打印每一层 shape。

## 概念问答

**Q：卷积层为什么适合图像？**  
A：它利用局部连接和参数共享，更符合图像局部结构。

**Q：池化会不会丢信息？**  
A：会减少细节，但也降低计算量，并保留较强响应。

**Q：为什么 feature map 数量会变多？**  
A：每个输出通道可以学习一种不同的局部模式。

**Q：小图像也需要 CNN 吗？**  
A：不一定，但 CNN 能展示图像建模的基本思路。

## 小结

CNN 的核心是用卷积层学习空间局部模式，用池化降低尺寸，再用分类头输出类别。写 CNN 时不要急着堆层，先把每一步 shape 走通。